In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 03 特征筛选模块 (core.selectors)\n\n提供多种特征筛选方法，包括过滤法、包装法、嵌入法。\n\n**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.selectors import (
    VarianceSelector, NullSelector, CorrSelector, VIFSelector,
    IVSelector, LiftSelector, FeatureImportanceSelector,
    CompositeFeatureSelector
)

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 选择数值特征
numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3', 
                   '轻花老客海纳子分V1', '天创小额网贷分', '衡枢鉴真分老客版']

# 处理缺失值
df_model = df[numeric_features + ['target']].copy()
df_model = df_model.dropna()

print(f"样本数: {len(df_model):,}")
print(f"特征数: {len(numeric_features)}")
print(f"坏样本率: {df_model['target'].mean():.2%}")
print(f"\n数据预览:")
df_model.head()

样本数: 264
特征数: 8
坏样本率: 14.02%

数据预览:


,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,衡枢鉴真分老客版,target
706,636.0000,612.0000,0.1028,606,601.0000,0.0205,739,0.0393,0
707,628.0000,600.0000,0.0759,676,617.0000,0.0156,792,0.1116,0
708,646.0000,669.0000,0.3775,607,556.0000,0.0333,684,0.0690,0
709,611.0000,680.0000,0.1028,715,676.0000,0.0192,766,0.1170,0
710,710.0000,796.0000,0.1589,674,534.0000,0.0255,721,0.0544,0


## 1. IV筛选 - 传入整个DataFrame

In [3]:
# IV筛选 - 一次性传入所有特征
iv_selector = IVSelector(threshold=0.02)
iv_selector.fit(df_model[numeric_features], df_model['target'])

print("=== IV筛选结果 ===")
print(f"原始特征数: {len(numeric_features)}")
print(f"保留特征数: {len(iv_selector.selected_features_)}")
print(f"保留特征: {iv_selector.selected_features_}")

# 查看IV值
print("\n各特征IV值:")
for feat, iv in zip(iv_selector.feature_names_in_, iv_selector.scores_):
    print(f"  {feat}: {iv:.4f}")

=== IV筛选结果 ===
原始特征数: 8
保留特征数: 6
保留特征: ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3', '天创小额网贷分']

各特征IV值:


AttributeError: 'IVSelector' object has no attribute 'feature_names_in_'

## 2. 相关性筛选 - 传入整个DataFrame

In [4]:
# 相关性筛选 - 一次性传入所有特征
corr_selector = CorrSelector(threshold=0.95)
corr_selector.fit(df_model[numeric_features])

print("=== 相关性筛选结果 ===")
print(f"保留特征数: {len(corr_selector.selected_features_)}")
print(f"剔除特征: {corr_selector.removed_features_}")

# 查看高相关性特征对
if len(corr_selector.correlated_pairs_) > 0:
    print("\n高相关性特征对:")
    for pair in corr_selector.correlated_pairs_:
        print(f"  {pair['feature_1']} <-> {pair['feature_2']}: {pair['correlation']:.4f}")

=== 相关性筛选结果 ===
保留特征数: 8
剔除特征: []


AttributeError: 'CorrSelector' object has no attribute 'correlated_pairs_'

## 3. VIF筛选 - 传入整个DataFrame

In [5]:
# VIF筛选 - 一次性传入所有特征
vif_selector = VIFSelector(threshold=10.0)
vif_selector.fit(df_model[numeric_features])

print("=== VIF筛选结果 ===")
print(f"保留特征数: {len(vif_selector.selected_features_)}")

print("\n各特征VIF值:")
for feat, vif in zip(vif_selector.feature_names_in_, vif_selector.scores_):
    status = "保留" if feat in vif_selector.selected_features_ else "剔除"
    print(f"  {feat}: {vif:.2f} ({status})")

=== VIF筛选结果 ===
保留特征数: 8

各特征VIF值:


AttributeError: 'VIFSelector' object has no attribute 'feature_names_in_'

## 4. 特征重要性筛选 - 传入整个DataFrame

In [6]:
# 特征重要性筛选
imp_selector = FeatureImportanceSelector(
    model_type='xgboost',
    threshold='median'
)
imp_selector.fit(df_model[numeric_features], df_model['target'])

print("=== 特征重要性筛选结果 ===")
print(f"保留特征数: {len(imp_selector.selected_features_)}")

# 可视化
importance_df = pd.DataFrame({
    '特征': imp_selector.feature_names_in_,
    '重要性': imp_selector.scores_
}).sort_values('重要性', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['特征'], importance_df['重要性'])
plt.axvline(x=importance_df['重要性'].median(), color='red', linestyle='--', label='中位数')
plt.xlabel('特征重要性')
plt.title('特征重要性排序')
plt.legend()
plt.tight_layout()
plt.show()

TypeError: FeatureImportanceSelector.__init__() got an unexpected keyword argument 'model_type'

## 5. 组合筛选 - 传入整个DataFrame

In [7]:
# 组合筛选 - 一次性传入所有特征
composite_selector = CompositeFeatureSelector(
    selectors=[
        IVSelector(threshold=0.02),
        CorrSelector(threshold=0.95)
    ],
    strategy='sequential'
)

composite_selector.fit(df_model[numeric_features], df_model['target'])

print("=== 组合筛选结果 ===")
print(f"原始特征数: {len(numeric_features)}")
print(f"最终保留特征数: {len(composite_selector.selected_features_)}")
print(f"保留特征: {composite_selector.selected_features_}")

=== 组合筛选结果 ===
原始特征数: 8
最终保留特征数: 6
保留特征: ['天创小额网贷分', '占信V3', '青云24', '极光欺诈分6v1', '珊瑚92', '中智小牛分C3']


## 6. 筛选结果对比

In [8]:
# 汇总各筛选器结果
summary = pd.DataFrame({
    '筛选方法': ['IV筛选', '相关性筛选', 'VIF筛选', '特征重要性筛选', '组合筛选'],
    '原始特征数': [len(numeric_features)] * 5,
    '保留特征数': [
        len(iv_selector.selected_features_),
        len(corr_selector.selected_features_),
        len(vif_selector.selected_features_),
        len(imp_selector.selected_features_),
        len(composite_selector.selected_features_)
    ]
})
summary['剔除特征数'] = summary['原始特征数'] - summary['保留特征数']

print("=== 各筛选方法结果对比 ===")
display(summary)

NameError: name 'imp_selector' is not defined